# The TaskScheduler: Task Distribution and Data Locality

**Module**: 04_execution_engine  
**Notebook**: 04_task_scheduler.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How the TaskScheduler dispatches TaskSets to executors, evaluates data locality, and handles speculative execution.
- **Why does this topic exist?**: TaskScheduler handles low-level thread scheduling and core utilization.
- **What problem does it solve?**: Mitigating stragglers and node failures.
- **Where is it used in real-world Data Engineering?**: Cluster resource tuning.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DAGScheduler, cluster architecture |
| **Required depth** | Deep internals: can explain task assignment, locality, and retries |
| **Data Engineer** | Read task-level metrics and understand locality and retry behavior. |
| **Senior Data Engineer** | Tune locality wait, speculation, and executor resources. |
| **Spark specialist** | Explain SchedulerBackend offers, FIFO/FAIR pools, and task failure handling. |
| **Next topic** | `04_execution_engine/05_jobs_stages_and_tasks.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The TaskScheduler decides which executor gets which task and when. It balances locality, resource
availability, and fairness. A task can be pending because no executor has a free core, because locality waits are
too long, or because the scheduler backend cannot offer resources. These are different problems with different
fixes.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **TaskSet** | Tasks for one stage | One task per partition |
| **Locality** | Data proximity | PROCESS_LOCAL, NODE_LOCAL, RACK_LOCAL, ANY |
| **SchedulerBackend** | Talks to the cluster manager | Offers resources and launches tasks |
| **FIFO/FAIR** | Scheduling policy | Queue/pool behavior |
| **Speculation** | Duplicate slow tasks | Mitigates stragglers |
| **Task retry** | Re-run a failed task | `spark.task.maxFailures` |

### How it works, step by step

1. The DAGScheduler submits a TaskSet.
2. The TaskScheduler asks the backend for resource offers.
3. It matches tasks to executors by locality and availability.
4. Tasks are serialized and launched on executor threads.
5. Completed/failed status updates return to the driver.
6. Failed tasks are retried; speculative copies may be launched for stragglers.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. TaskScheduler Locality Levels: PROCESS_LOCAL > NODE_LOCAL > NO_PREF > RACK_LOCAL > ANY

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("TaskScheduler_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rdd = spark.sparkContext.parallelize(range(100), 4)
print("Tasks run across available worker cores:")
print(rdd.map(lambda x: x*2).collect()[:10])

---

## Experiment: Speculative Execution Flag

- **1. What are we testing?**: Check speculation config.
- **2. What do we expect?**: False by default.

In [ ]:
print('Speculation enabled?:', spark.conf.get('spark.speculation', 'false'))

**3. What happened?**: Printed false.  
**4. Why did it happen?**: Speculation runs duplicate tasks for stragglers.  
**5. Practical Engineering Lesson**: Enable speculation on large shared clusters with flaky nodes.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Locality tuning for a data lake

- **Industry**: Media
- **Scenario**: A Spark job reads large Parquet files from shared storage and tasks wait for locality.
- **Problem**: The default locality wait adds latency without improving data proximity because the storage is remote.
- **Why this concept is relevant**: Reducing locality wait can improve runtime when data is not co-located with compute.
- **Implementation idea**: Measure task launch delay and locality levels; tune `spark.locality.wait` for the storage architecture.
- **Expected result**: Lower latency without hurting throughput.
- **Alternative**: A managed Spark service with adaptive locality.
- **Trade-offs**: Local disks/HDFS improve locality; object storage changes the trade-off.

### Use Case 2: Straggler mitigation

- **Industry**: Advertising
- **Scenario**: A few tasks in a large aggregation run much longer than the rest.
- **Problem**: The job waits for the slowest task even though most executors are idle.
- **Why this concept is relevant**: Speculative execution can launch a duplicate attempt and take the first successful result.
- **Implementation idea**: Enable speculation for the stage, inspect task duration percentiles, and fix the underlying skew if it recurs.
- **Expected result**: A shorter tail latency, with extra resource cost.
- **Alternative**: AQE skew join or salting.
- **Trade-offs**: Speculation is a band-aid; skew remediation addresses the cause.

---

## Business Value

- Improves resource utilization and tail latency.
- Reduces time lost to a single slow node.
- Helps teams choose between locality and latency.
- Supports more predictable SLAs.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Monitor pending tasks and locality levels in the Stages tab.
- Do not enable speculation blindly; it consumes extra resources.
- Fix skew before relying on speculation.
- Check `spark.task.maxFailures` and executor loss behavior.
- Use FAIR pools when multiple users share a cluster.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Locality levels and wait time

**Steps**

1. Run a job reading Parquet and inspect task locality in the Stages tab.
2. Set a very low `spark.locality.wait` and compare task launch time.
3. Record whether data locality actually improved.

**What to observe**: Locality waits trade launch delay for data proximity; remote storage may not benefit.

**Healthy result**: The chosen wait matches the storage/compute layout.

**Problematic result**: Long locality waits on object storage add latency without reducing I/O.

**Engineering takeaway**: Measure locality, do not assume it.

---

## Trade-offs and Alternatives

### Advantages

- Locality-aware
- Retry/failure handling
- Supports speculation and fairness

### Limitations

- Locality waits can hurt remote-storage workloads
- Speculation adds cost
- Tuning is workload-specific

### When to use it

- Task-level debugging
- Straggler mitigation
- Shared clusters

### When not to use it

- Speculation as a substitute for skew fixes
- Assuming locality always helps object storage

### Alternatives inside Spark

- DAGScheduler
- Dynamic allocation
- AQE

### Alternatives outside Spark

- Kubernetes scheduler
- YARN fair scheduler

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Default Parallelism
**Problem Statement**: Print sc.defaultParallelism.  
**Input / Context**: sc

In [ ]:
# B

### Exercise 2 (Intermediate): Task Attempt ID
**Problem Statement**: Inspect task attempt context.  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): Locality Inspection
**Problem Statement**: Check locality in UI.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Production Debugging Exercise

### Tasks pending with free cores

**Scenario**: The cluster has free cores but tasks remain pending.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the scheduler backend has registered the resources.
2. Inspect locality wait and executor registration.
3. Look for a driver/executor communication issue.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Default Parallelism):
```python
print(spark.sparkContext.defaultParallelism)
```

#### Solution 2 (Task Attempt ID):
```python
from pyspark import TaskContext
print(spark.sparkContext.parallelize(range(2)).map(lambda _: TaskContext.get().attemptNumber()).collect())
```

#### Solution 3 (Locality Inspection):
```python
print('Inspected task locality in UI')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **TaskScheduler**: Assigns tasks to executor threads based on locality.

### Essential Technical Terms
- `TaskSet`: Set of tasks for 1 stage.
- `Speculation`: Duplicate copy for slow tasks.

### Common Engineering Mistakes
- **Ignoring locality**: Remote data transfer slows tasks.

### Interview Questions on This Topic
- **Q: What is speculative execution?**
  - *A*: Launching a duplicate task on another worker if a task lags behind.


### More Interview Questions

- **Q: What is the difference between the DAGScheduler and the TaskScheduler?**
  - *A*: The DAGScheduler splits a job into stages; the TaskScheduler assigns the tasks of a stage to available executors and handles locality/retries.
- **Q: What are Spark's locality levels?**
  - *A*: PROCESS_LOCAL, NODE_LOCAL, RACK_LOCAL, and ANY, in decreasing order of proximity.

### What to Study Next
Open 05_jobs_stages_and_tasks.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')